In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset,DataLoader

from tqdm import tqdm

SR = 22050
DURATION = 30
model_name = "cnn"
try:
    dir_path = f"/kaggle/working/{model_name}"
    os.makedirs(dir_path, exist_ok=True)
    print(f"Directory created at: {dir_path}")
except Exception as e:
    print(f"Error creating directory: {e}")

#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")

Directory created at: /kaggle/working/cnn
🚀 Using device: cuda
GPU is available.Setting RANDOM_SEED .... 
   GPU: Tesla T4
   Memory: 14.6 GB

✅ Environment setup complete!


Your First Neural Network & CNNs!

* Learn PyTorch basics: Tensors, Dataset (custom loader for training), DataLoader.
* Convert audio to 2D/1D Mel-Spectrograms.
* Build a simple CNN (Convolutional Neural Network)/NN (Neural Network) to process the spectrograms.
* Implement training loop, loss, optimizer, and wandb logging.
* Train and evaluate your CNN/NN (Neural Network)model.

# Definition

## 1. Utility

In [2]:
def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    waveform, _sr_ = torchaudio.load(path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if _sr_ != sr:
        resampler = torchaudio.transforms.Resample(_sr_, sr)
        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)

    # Trim or pad
    if waveform.shape[0] >= LENGTH:
        return waveform[:LENGTH],sr
    else:
        padding = LENGTH - waveform.shape[0]
        return torch.nn.functional.pad(waveform, (0, padding)), sr

def extract_paths(pc=15000,test_size=0.2):
    GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
    train_paths = []
    val_paths = []
    count = 0
    tr_count = 0
    val_count = 0
    
    paths_count = pc
    test_size = paths_count*test_size
    train_size = paths_count - test_size
    
    for g in GENRES:
        root_dir_path = f"/kaggle/input/datasets/akashkumbhakar/{g}-15000"
        for i in range(0,paths_count):
            file_name = f"mashup_{i}.wav"
            path = os.path.join(root_dir_path,file_name)
            if i >= train_size:
                val_paths.append((path,g))
                val_count += 1
            else : 
                train_paths.append((path,g))
                tr_count += 1
            count += 1
    print("Total paths (music files) : ", count)
    print("Total training files : ", tr_count)
    print("Total validation files : ", val_count)
    return train_paths,val_paths

def check_split():
    GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
    g_c = {}
    for i in tr:
        if i[1] in g_c.keys():
            g_c[i[1]] += 1
        else:
            g_c[i[1]] = 1
    return g_c

def genre_to_idx(targets):
    genre_to_id = {'blues':0, 'classical':1, 'country':2, 'disco':3, 'hiphop':4,'jazz':5, 'metal':6, 'pop':7, 'reggae':8, 'rock':9}
    y = torch.tensor([genre_to_id[g] for g in targets])

    return y

def idx_to_genre(targets):
    id_to_genre = {0:'blues', 1:'classical', 2:'country', 3:'disco', 4:'hiphop',5:'jazz', 6:'metal', 7:'pop', 8:'reggae', 9:'rock'}
    y = [id_to_genre[id] for id in targets]

    return y    

## 2. Dataset and DataLoader

In [10]:
class MelDataset(Dataset):
    def __init__(self,paths):  # paths : List[(path,label)]
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self,idx):
        waveform, sr = load_and_fix(self.paths[idx][0],sr=SR,duration=DURATION)
        return waveform,self.paths[idx][1]

config = {
    "batch_size" : 128
}

mel_transform = T.MelSpectrogram(
    sample_rate = 22050,
    n_fft = 1024,
    n_mels=128
).to(device)

amplitude_to_db = T.AmplitudeToDB().to(device)

train_paths,val_paths = extract_paths()

train_dataset = MelDataset(train_paths)
val_dataset = MelDataset(val_paths)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

print(f"Size of Train Dataloader : {len(train_loader)}  |  Size of Val Dataloader : {len(val_loader)}")
print("✅")

Total paths (music files) :  150000
Total training files :  120000
Total validation files :  30000
Size of Train Dataloader : 938  |  Size of Val Dataloader : 235
✅


## 3. Model

In [19]:
%%writefile /kaggle/working/cnn/model.py
class MelCNN(nn.Module):
    def __init__(self, num_classes):
        super(MelCNN, self).__init__()
        # First convolutional block
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):

        x = x.unsqueeze(1)  # (B,1,128,1292)

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

Writing /kaggle/working/cnn/model.py


## 4. Training

In [12]:
def train(model,train_loader,loss_fn, optimizer, num_epochs=5):
    train_loss = []
    train_f1_score = []

    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader),desc=f"Epoch {epoch+1}/{num_epochs}")
        for i,(waveforms,labels) in progress_bar:
            waveforms , labels = waveforms.to(device,non_blocking=True), genre_to_idx(labels).to(device,non_blocking=True)
            optimizer.zero_grad()

            mels = mel_transform(waveforms)
            mels = amplitude_to_db(mels)
            
            outputs = model(mels)
            loss = loss_fn(outputs, labels)
        
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            avg_loss = running_loss / (i + 1)

            progress_bar.set_postfix({
                'loss': f"{avg_loss:.4f}"
            })
        loss = running_loss / len(train_loader)
        print(f"Train Loss: {loss:.4f}")

## 5. Evaluation

In [13]:
def validation(model,val_loader,loss_fn):
    model.eval()
    val_loss = 0.0
    all_true = []
    all_pred = []
    with torch.no_grad():
        for waveforms,labels in tqdm(val_loader, desc="Evaluating"):
            waveforms,labels = waveforms.to(device,non_blocking=True),genre_to_idx(labels).to(device,non_blocking=True)
            
            mels = mel_transform(waveforms)
            mels = amplitude_to_db(mels)
            
            output = model(mels)
            loss = loss_fn(output,labels)

            probs = torch.softmax(output,dim=1) 
            predicted_y = torch.argmax(probs,dim=1)
            
            val_loss += loss.item()

            all_true.append(labels)
            all_pred.append(predicted_y)
        all_true = torch.cat(all_true,dim=0).cpu().numpy()
        all_pred = torch.cat(all_pred,dim=0).cpu().numpy()
        print(all_true.shape,all_pred.shape)
    
        loss = val_loss/len(val_loader)
        f1score = f1_score(all_true,all_pred,average='macro')

        print(f"Validation Loss : {loss},  F1_score : {f1score}")
        return all_true,all_pred
            

# Training

In [14]:
model = MelCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


train(model,train_loader,criterion, optimizer, num_epochs=5)

Epoch 1/5: 100%|██████████| 938/938 [26:18<00:00,  1.68s/it, loss=0.9777]


Train Loss: 0.9777


Epoch 2/5: 100%|██████████| 938/938 [24:58<00:00,  1.60s/it, loss=0.4571]


Train Loss: 0.4571


Epoch 3/5: 100%|██████████| 938/938 [24:41<00:00,  1.58s/it, loss=0.3282]


Train Loss: 0.3282


Epoch 4/5: 100%|██████████| 938/938 [24:12<00:00,  1.55s/it, loss=0.2676]


Train Loss: 0.2676


Epoch 5/5: 100%|██████████| 938/938 [24:42<00:00,  1.58s/it, loss=0.2230]

Train Loss: 0.2230


In [15]:
all_true,all_pred = validation(model,val_loader,criterion)

Evaluating: 100%|██████████| 235/235 [07:10<00:00,  1.83s/it]

(30000,) (30000,)
Validation Loss : 0.358534736772801,  F1_score : 0.8680990737988348


# Saving model

In [20]:
if os.path.exists(f"/kaggle/working/{model_name}"):
    torch.save(model.state_dict(), f"/kaggle/working/{model_name}/model.pt")
    print(f"✅ {model_name} saved.")
else:
    print(f"Path not exists.")

✅ cnn saved.


# Uploading to KaggleHub

In [21]:
import kagglehub

# Replace with path to directory containing model files.
LOCAL_MODEL_DIR = f'/kaggle/working/{model_name}'

MODEL_SLUG = model_name # Replace with model slug.

# Learn more about naming model variations at
# https://www.kaggle.com/docs/models#name-model.
VARIATION_SLUG = 'default' # Replace with variation slug.

kagglehub.model_upload(
  handle = f"akashkumbhakar/{MODEL_SLUG}/pyTorch/{VARIATION_SLUG}",
  local_model_dir = LOCAL_MODEL_DIR,
  version_notes = 'Update 2026-03-08')

Uploading Model https://api.kaggle.com/models/akashkumbhakar/cnn/pyTorch/default ...
Model 'cnn' does not exist or access is forbidden for user 'akashkumbhakar'. Creating or handling Model...
Model 'cnn' Created.
Starting upload for file /kaggle/working/cnn/model.pt


Uploading: 100%|██████████| 104k/104k [00:00<00:00, 217kB/s]

Upload successful: /kaggle/working/cnn/model.pt (102KB)
Starting upload for file /kaggle/working/cnn/model.py



Uploading: 100%|██████████| 936/936 [00:00<00:00, 2.31kB/s]

Upload successful: /kaggle/working/cnn/model.py (936B)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/akashkumbhakar/cnn/pyTorch/default
